# Hugging Face Transformers 微调训练入门

本示例将介绍基于 Transformers 实现模型微调训练的主要流程，包括：
- 数据集下载
- 数据预处理
- 训练超参数配置
- 训练评估指标设置
- 训练器基本介绍
- 实战训练
- 模型保存

## YelpReviewFull 数据集

**Hugging Face 数据集：[ YelpReviewFull ](https://huggingface.co/datasets/Yelp/yelp_review_full)**

### 数据集摘要

Yelp评论数据集包括来自Yelp的评论。它是从Yelp Dataset Challenge 2015数据中提取的。

### 支持的任务和排行榜
文本分类、情感分类：该数据集主要用于文本分类：给定文本，预测情感。

### 语言
这些评论主要以英语编写。

### 数据集结构

#### 数据实例
一个典型的数据点包括文本和相应的标签。

来自YelpReviewFull测试集的示例如下：

```json
{
    'label': 0,
    'text': 'I got \'new\' tires from them and within two weeks got a flat. I took my car to a local mechanic to see if i could get the hole patched, but they said the reason I had a flat was because the previous patch had blown - WAIT, WHAT? I just got the tire and never needed to have it patched? This was supposed to be a new tire. \\nI took the tire over to Flynn\'s and they told me that someone punctured my tire, then tried to patch it. So there are resentful tire slashers? I find that very unlikely. After arguing with the guy and telling him that his logic was far fetched he said he\'d give me a new tire \\"this time\\". \\nI will never go back to Flynn\'s b/c of the way this guy treated me and the simple fact that they gave me a used tire!'
}
```

#### 数据字段

- 'text': 评论文本使用双引号（"）转义，任何内部双引号都通过2个双引号（""）转义。换行符使用反斜杠后跟一个 "n" 字符转义，即 "\n"。
- 'label': 对应于评论的分数（介于1和5之间）。

#### 数据拆分

Yelp评论完整星级数据集是通过随机选取每个1到5星评论的130,000个训练样本和10,000个测试样本构建的。总共有650,000个训练样本和50,000个测试样本。

## 下载数据集

In [ ]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [ ]:
dataset["train"][111]

{'label': 2,
 'text': "As far as Starbucks go, this is a pretty nice one.  The baristas are friendly and while I was here, a lot of regulars must have come in, because they bantered away with almost everyone.  The bathroom was clean and well maintained and the trash wasn't overflowing in the canisters around the store.  The pastries looked fresh, but I didn't partake.  The noise level was also at a nice working level - not too loud, music just barely audible.\\n\\nI do wish there was more seating.  It is nice that this location has a counter at the end of the bar for sole workers, but it doesn't replace more tables.  I'm sure this isn't as much of a problem in the summer when there's the space outside.\\n\\nThere was a treat receipt promo going on, but the barista didn't tell me about it, which I found odd.  Usually when they have promos like that going on, they ask everyone if they want their receipt to come back later in the day to claim whatever the offer is.  Today it was one of th

In [ ]:
import random
import pandas as pd
import datasets
from IPython.display import display, HTML

In [ ]:
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [ ]:
show_random_elements(dataset["train"])

,label,text
0,3 stars,"I usually get steak n eggs at south point late at night but we decided to try this place since their steak n eggs special is all day. For $5 you get a steak, hash browns, eggs, and toast. The steak was really thin so you're bound to finish it pretty fast. Everything was worth the price paid for but it could be a little better!\n\nThe service was pretty weird. I couldn't tell if our server was being sarcastic or a total asshole."
1,1 star,"We bought our home & moved into it July '11. The house had been vacant for approximately 10 months but pest service was maintained by the seller. After just a few days, we had a cold-call salesman selling us their service. \n\nHusband agreed - stating that the previous owners disclosed \""scorpions\"" and we had seen scorpions, crickets & spiders - all dead but evidence that they were there & being controlled by prior service. We set up a \""date\"" four our initial treatment. \n\nLess than a month later, I found a HUGE spider - quite alive. When I smashed it, she erupted into a flood of a million little tiny spiders. *gag* \n\nA week later, another spider.\n\nA week later, a scorpion - quite alive.\n\nWe were \""retreated\"" where my husband followed the guy around - showing him where to spray. Hundreds of dead crickets were the result of the respray - but not a scorpion or spider corpse to be found. \n\nA couple weeks later... more spiders. More scorpions. ALIVE. Inside my house. In my child's bed... in the upstairs bathroom... in the downstairs bathroom... in the kitchen... in the garage. \n\nWe called back to cancel service & were told, \""Oh... you've never gotten a scorpion treatment. You have to specifically ASK for those...\"" \n\n(Apparently, letting the initial guy know that scorpions were our MAIN point of concern wasn't enough. We had to ask \""pretty please will you spray for scorpions, too?!\"")\n\nWe were also told, \""Well... this IS Arizona...\"" \n\nOk - so because this is Arizona means you don't have to stand behind your service? You are selling me pest control. I expect you to control the pests. Finding scorpions in my kids beds is NOT under control. If I pay you to kill bugs, I do not expect live scorpions in my house at regular & frequent intervals.\n\nps - I live in a well populated sub-division... not in a rural area.\n\nWe scheduled another re-spray... this time including a scorpion service.\n\nYep... you guessed it... a couple weeks later...\n\nMore scorpions & spiders IN THE HOUSE. \n\nMy husband called - again - to cancel the service and now they are threatening us with a \""early termination fee\"" because we have only been customers for 4 months. Apparently, we are being held to a minimum contract term that was never communicated.\n\nThey will review our contract and get back to us tomorrow... if they do not let us cancel this \""understood\"" contract for services that are NOT being provided, they're going to be seeing more of me than they will care to... I won't hesitate to become the thorn in their side scheduling weekly resprays. \n\nIn summary, awful service for ineffective pest service."
2,1 star,"Do yourself a favor and go elsewhere! 3 hours for a \""partial highlight\"" I went running out of the salon with wet hair! Worst experience in a salon ever..."
3,3 stars,"Come because you have a coupon. Stay because there's a lot of beer.\n\nThis faux English pub is nestled in the corner of Riviera's casino floor. I was excited to go there because there was a buy one get one coupon. There are actually coupons for this place all around the Riviera. Make sure you grab some before you head to the pub. Because the beer prices are pretty high. The average beer is around $7 to $8. So you can cut that in half with a coupon. There are around 100 different draft and bottle choices. Obviously a lot of English beer choices. I'm not a big fan of dark heavy beer, but that's what was mostly available. Also, the advertisements make it seem like this bar ha

## 预处理数据

下载数据集到本地后，使用 Tokenizer 来处理文本，对于长度不等的输入数据，可以使用填充（padding）和截断（truncation）策略来处理。

Datasets 的 `map` 方法，支持一次性在整个数据集上应用预处理函数。

下面使用填充到最大长度的策略，处理整个数据集：

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

/Users/xizhi/Code/LLM-quickstart/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [8]:
show_random_elements(tokenized_datasets["train"], num_examples=1)

,label,text,input_ids,token_type_ids,attention_mask
0,3 stars,"I like this store but their female help in the upstairs needs improvement. Tell marnetta to bring more of the cute girls from downstairs up there! Need that to round out the \""upscale\"" shopping experience. Also cute girls nicer in general whereas homely people are cranky and self-loathing and high maintenance.\n\n*update*\nSee Rachel's review several months after mine. Wadya know. She also has experienced the fine service of local proleteriat women. Hey rachel, was she homely?","[101, 146, 1176, 1142, 2984, 1133, 1147, 2130, 1494, 1107, 1103, 8829, 2993, 8331, 119, 4630, 12477, 12275, 5100, 1106, 2498, 1167, 1104, 1103, 10509, 2636, 1121, 10304, 1146, 1175, 106, 12528, 1115, 1106, 1668, 1149, 1103, 165, 107, 12534, 20532, 165, 107, 6001, 2541, 119, 2907, 10509, 2636, 3505, 1197, 1107, 1704, 6142, 1313, 1193, 1234, 1132, 172, 14687, 1183, 1105, 2191, 118, 25338, 9779, 1158, 1105, 1344, 5972, 119, 165, 183, 165, 183, 115, 11984, 115, 165, 183, 1708, 3051, 4858, 112, 188, 3189, 1317, 1808, 1170, 2317, 119, 160, 16196, 1161, 1221, 119, 1153, 1145, 1144, 4531, ...]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...]"


### 数据抽样

使用 1000 个数据样本，在 BERT 上演示小规模训练（基于 Pytorch Trainer）

`shuffle()`函数会随机重新排列列的值。如果您希望对用于洗牌数据集的算法有更多控制，可以在此函数中指定generator参数来使用不同的numpy.random.Generator。

In [9]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

## 微调训练配置

### 加载 BERT 模型

警告通知我们正在丢弃一些权重（`vocab_transform` 和 `vocab_layer_norm` 层），并随机初始化其他一些权重（`pre_classifier` 和 `classifier` 层）。在微调模型情况下是绝对正常的，因为我们正在删除用于预训练模型的掩码语言建模任务的头部，并用一个新的头部替换它，对于这个新头部，我们没有预训练的权重，所以库会警告我们在用它进行推理之前应该对这个模型进行微调，而这正是我们要做的事情。

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

/Users/xizhi/Code/LLM-quickstart/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 训练超参数（TrainingArguments）

完整配置参数与默认值：https://huggingface.co/docs/transformers/v4.36.1/en/main_classes/trainer#transformers.TrainingArguments

源代码定义：https://github.com/huggingface/transformers/blob/v4.36.1/src/transformers/training_args.py#L161

**最重要配置：模型权重保存路径(output_dir)**

In [11]:
from transformers import TrainingArguments

model_dir = "models/bert-base-cased-finetune-yelp"

# logging_steps 默认值为500，根据我们的训练数据和步长，将其设置为100
training_args = TrainingArguments(output_dir=model_dir,
                                  per_device_train_batch_size=16,
                                  num_train_epochs=5,
                                  logging_steps=100)

In [12]:
# 完整的超参数配置
print(training_args)

TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_steps=None,
evaluation_strategy=no,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'min_num_params': 0, 'xla': False, 'xla_fsdp_grad_ckpt': False},
fsdp_min_num_params=0,
fsdp_transformer_layer_cls_to_wrap=None,
full_determinism=False,
gradient_accumulation_steps=1,
gradient_checkpointing=False,
gradient_checkpointing_kwargs=None,
greater_is_better=None,
group_by_le

### 训练过程中的指标评估（Evaluate)

**[Hugging Face Evaluate 库](https://huggingface.co/docs/evaluate/index)** 支持使用一行代码，获得数十种不同领域（自然语言处理、计算机视觉、强化学习等）的评估方法。 当前支持 **完整评估指标：https://huggingface.co/evaluate-metric**

训练器（Trainer）在训练过程中不会自动评估模型性能。因此，我们需要向训练器传递一个函数来计算和报告指标。 

Evaluate库提供了一个简单的准确率函数，您可以使用`evaluate.load`函数加载

In [13]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")


接着，调用 `compute` 函数来计算预测的准确率。

在将预测传递给 compute 函数之前，我们需要将 logits 转换为预测值（**所有Transformers 模型都返回 logits**）。

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

#### 训练过程指标监控

通常，为了监控训练过程中的评估指标变化，我们可以在`TrainingArguments`指定`evaluation_strategy`参数，以便在 epoch 结束时报告评估指标。

In [15]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir=model_dir,
                                  evaluation_strategy="epoch", 
                                  per_device_train_batch_size=16,
                                  num_train_epochs=3,
                                  logging_steps=30)

## 开始训练

### 实例化训练器（Trainer）

`kernel version` 版本问题：暂不影响本示例代码运行

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

## 使用 nvidia-smi 查看 GPU 使用

为了实时查看GPU使用情况，可以使用 `watch` 指令实现轮询：`watch -n 1 nvidia-smi`:

```shell
Every 1.0s: nvidia-smi                                                   Wed Dec 20 14:37:41 2023

Wed Dec 20 14:37:41 2023
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   64C    P0              69W /  70W |   6665MiB / 15360MiB |     98%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A     18395      C   /root/miniconda3/bin/python                6660MiB |
+---------------------------------------------------------------------------------------+
```

In [17]:
trainer.train()

  0%|          | 0/189 [00:00<?, ?it/s]

{'loss': 1.6203, 'learning_rate': 4.2063492063492065e-05, 'epoch': 0.48}
{'loss': 1.3623, 'learning_rate': 3.412698412698413e-05, 'epoch': 0.95}


  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 1.1137257814407349, 'eval_accuracy': 0.553, 'eval_runtime': 11.1264, 'eval_samples_per_second': 89.877, 'eval_steps_per_second': 11.235, 'epoch': 1.0}
{'loss': 1.0364, 'learning_rate': 2.6190476190476192e-05, 'epoch': 1.43}
{'loss': 0.9172, 'learning_rate': 1.8253968253968254e-05, 'epoch': 1.9}


  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 1.0085304975509644, 'eval_accuracy': 0.574, 'eval_runtime': 10.3144, 'eval_samples_per_second': 96.952, 'eval_steps_per_second': 12.119, 'epoch': 2.0}
{'loss': 0.7072, 'learning_rate': 1.0317460317460318e-05, 'epoch': 2.38}
{'loss': 0.6719, 'learning_rate': 2.3809523809523808e-06, 'epoch': 2.86}


  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 1.0103576183319092, 'eval_accuracy': 0.585, 'eval_runtime': 10.3888, 'eval_samples_per_second': 96.258, 'eval_steps_per_second': 12.032, 'epoch': 3.0}
{'train_runtime': 141.8921, 'train_samples_per_second': 21.143, 'train_steps_per_second': 1.332, 'train_loss': 1.0322908345984403, 'epoch': 3.0}


TrainOutput(global_step=189, training_loss=1.0322908345984403, metrics={'train_runtime': 141.8921, 'train_samples_per_second': 21.143, 'train_steps_per_second': 1.332, 'train_loss': 1.0322908345984403, 'epoch': 3.0})

In [18]:
small_test_dataset = tokenized_datasets["test"].shuffle(seed=64).select(range(100))

In [19]:
trainer.evaluate(small_test_dataset)

  0%|          | 0/13 [00:00<?, ?it/s]

{'eval_loss': 1.048350214958191,
 'eval_accuracy': 0.53,
 'eval_runtime': 1.3523,
 'eval_samples_per_second': 73.946,
 'eval_steps_per_second': 9.613,
 'epoch': 3.0}

### 保存模型和训练状态

- 使用 `trainer.save_model` 方法保存模型，后续可以通过 from_pretrained() 方法重新加载
- 使用 `trainer.save_state` 方法保存训练状态

In [20]:
trainer.save_model(model_dir)

In [21]:
trainer.save_state()

In [23]:
# trainer.model.save_pretrained("./")

## Homework: 使用完整的 YelpReviewFull 数据集训练，看 Acc 最高能到多少

In [24]:
# Homework：在「完整」YelpReviewFull 上训练（train 650k / test 50k），观察 eval_accuracy
# 依赖上文已执行：load_dataset → map 得到 tokenized_datasets，以及 compute_metrics
# 完整训练耗时长、占显存大，建议在 NVIDIA GPU 上跑；显存不够可把 batch 调小或减小 num_train_epochs

import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model_dir_full = "models/bert-base-cased-finetune-yelp-full"

full_train_dataset = tokenized_datasets["train"]
full_eval_dataset = tokenized_datasets["test"]

# 使用新模型实例，避免在上文已训练过的 model 上继续训
model_full = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased", num_labels=5
)

training_args_full = TrainingArguments(
    output_dir=model_dir_full,  # 训练输出目录：权重、checkpoint、TensorBoard 日志等
    evaluation_strategy="epoch",  # 每个 epoch 结束在验证集上算 eval_loss / eval_accuracy
    save_strategy="epoch",  # 按 epoch 保存 checkpoint（与上面对齐）
    per_device_train_batch_size=16,  # 单卡上每个 step 的训练样本数；显存不够可改小
    per_device_eval_batch_size=32,  # 验证时可略大（不做反向传播）
    num_train_epochs=3,  # 全量数据扫 3 遍；越大越慢，可酌情增减
    logging_steps=500,  # 每 500 个优化 step 打一次训练 loss 日志
    load_best_model_at_end=True,  # 训练结束后加载「验证集上最优」那一轮权重，而非最后一轮
    metric_for_best_model="eval_accuracy",  # 以验证集准确率作为「最优」标准（Trainer 会加 eval_ 前缀）
    greater_is_better=True,  # 该指标越大越好（若是 loss 则要 False）
    save_total_limit=2,  # 磁盘上最多保留 2 个 checkpoint，避免占满空间
    fp16=torch.cuda.is_available(),  # 有 NVIDIA GPU 时用半精度加速并省显存；CPU/MPS 下为 False
    report_to=[],  # 不向 Weights & Biases 等上报；[] 表示关闭第三方实验跟踪
)

trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=full_train_dataset,
    eval_dataset=full_eval_dataset,
    compute_metrics=compute_metrics,
)

train_result = trainer_full.train()
print(train_result)

eval_metrics = trainer_full.evaluate()
print("Final eval:", eval_metrics)

trainer_full.save_model(model_dir_full)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/121875 [00:00<?, ?it/s]

{'loss': 1.1582, 'learning_rate': 4.979487179487179e-05, 'epoch': 0.01}
{'loss': 0.9818, 'learning_rate': 4.9589743589743594e-05, 'epoch': 0.02}
{'loss': 0.9547, 'learning_rate': 4.9384615384615384e-05, 'epoch': 0.04}
{'loss': 0.9355, 'learning_rate': 4.917948717948718e-05, 'epoch': 0.05}
{'loss': 0.9186, 'learning_rate': 4.8974358974358975e-05, 'epoch': 0.06}
{'loss': 0.8864, 'learning_rate': 4.876923076923077e-05, 'epoch': 0.07}
{'loss': 0.8924, 'learning_rate': 4.856410256410257e-05, 'epoch': 0.09}
{'loss': 0.8829, 'learning_rate': 4.835897435897436e-05, 'epoch': 0.1}
{'loss': 0.8822, 'learning_rate': 4.815384615384615e-05, 'epoch': 0.11}
{'loss': 0.8793, 'learning_rate': 4.7948717948717955e-05, 'epoch': 0.12}
{'loss': 0.8768, 'learning_rate': 4.7743589743589744e-05, 'epoch': 0.14}
{'loss': 0.8465, 'learning_rate': 4.753846153846154e-05, 'epoch': 0.15}
{'loss': 0.8528, 'learning_rate': 4.7333333333333336e-05, 'epoch': 0.16}
{'loss': 0.8698, 'learning_rate': 4.712820512820513e-05, 'e

  0%|          | 0/1563 [00:00<?, ?it/s]

{'eval_loss': 0.7657163143157959, 'eval_accuracy': 0.66896, 'eval_runtime': 457.8788, 'eval_samples_per_second': 109.199, 'eval_steps_per_second': 3.414, 'epoch': 1.0}
{'loss': 0.7442, 'learning_rate': 3.317948717948718e-05, 'epoch': 1.01}
{'loss': 0.7189, 'learning_rate': 3.297435897435898e-05, 'epoch': 1.02}
{'loss': 0.7179, 'learning_rate': 3.276923076923077e-05, 'epoch': 1.03}
{'loss': 0.7328, 'learning_rate': 3.2564102564102565e-05, 'epoch': 1.05}
{'loss': 0.7357, 'learning_rate': 3.235897435897436e-05, 'epoch': 1.06}
{'loss': 0.7351, 'learning_rate': 3.215384615384616e-05, 'epoch': 1.07}
{'loss': 0.7075, 'learning_rate': 3.1948717948717946e-05, 'epoch': 1.08}
{'loss': 0.7252, 'learning_rate': 3.174358974358975e-05, 'epoch': 1.1}
{'loss': 0.7126, 'learning_rate': 3.153846153846154e-05, 'epoch': 1.11}
{'loss': 0.7084, 'learning_rate': 3.1333333333333334e-05, 'epoch': 1.12}
{'loss': 0.7086, 'learning_rate': 3.112820512820513e-05, 'epoch': 1.13}
{'loss': 0.7194, 'learning_rate': 3.09

...

{'eval_loss': 0.7358123660087585, 'eval_accuracy': 0.68164, 'eval_runtime': 443.9769, 'eval_samples_per_second': 112.618, 'eval_steps_per_second': 3.52, 'epoch': 2.0}
{'loss': 0.6602, 'learning_rate': 1.6564102564102565e-05, 'epoch': 2.01}
{'loss': 0.6193, 'learning_rate': 1.635897435897436e-05, 'epoch': 2.02}
{'loss': 0.6236, 'learning_rate': 1.6153846153846154e-05, 'epoch': 2.03}
{'loss': 0.6299, 'learning_rate': 1.594871794871795e-05, 'epoch': 2.04}
{'loss': 0.6251, 'learning_rate': 1.5743589743589746e-05, 'epoch': 2.06}
{'loss': 0.6208, 'learning_rate': 1.553846153846154e-05, 'epoch': 2.07}
{'loss': 0.6238, 'learning_rate': 1.5333333333333334e-05, 'epoch': 2.08}
{'loss': 0.6287, 'learning_rate': 1.5128205128205129e-05, 'epoch': 2.09}
{'loss': 0.6239, 'learning_rate': 1.4923076923076923e-05, 'epoch': 2.1}
{'loss': 0.6309, 'learning_rate': 1.4717948717948717e-05, 'epoch': 2.12}
{'loss': 0.6318, 'learning_rate': 1.4512820512820513e-05, 'epoch': 2.13}
{'loss': 0.619, 'learning_rate': 1

  0%|          | 0/1563 [00:00<?, ?it/s]

{'eval_loss': 0.7411723732948303, 'eval_accuracy': 0.68976, 'eval_runtime': 458.3987, 'eval_samples_per_second': 109.075, 'eval_steps_per_second': 3.41, 'epoch': 3.0}
{'train_runtime': 65062.4703, 'train_samples_per_second': 29.971, 'train_steps_per_second': 1.873, 'train_loss': 0.7154422402093349, 'epoch': 3.0}
TrainOutput(global_step=121875, training_loss=0.7154422402093349, metrics={'train_runtime': 65062.4703, 'train_samples_per_second': 29.971, 'train_steps_per_second': 1.873, 'train_loss': 0.7154422402093349, 'epoch': 3.0})


  0%|          | 0/1563 [00:00<?, ?it/s]

Final eval: {'eval_loss': 0.7411723732948303, 'eval_accuracy': 0.68976, 'eval_runtime': 450.1852, 'eval_samples_per_second': 111.065, 'eval_steps_per_second': 3.472, 'epoch': 3.0}
